In [1]:
import os
import sys
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
plt.rcParams["figure.figsize"] = (24,18)
from utils import *

In [2]:
model_path = os.path.join(os.path.dirname(os.getcwd()), 'models')
model = YOLO(os.path.join(model_path, "yolov8n.pt"))
data_path = os.path.join(os.path.dirname(os.getcwd()), 'data')
vid_name =  'real_test'

In [3]:
# Get several images to test
images_path = os.path.join(data_path, 'images' + '/' + vid_name)  #print(os.path.exists(images_path))
first_batch = os.listdir(images_path)[50:55]
image_batch = [os.path.join(images_path, x) for x in first_batch]

# Attributes to identify colors
global_color_dict = {'team_1_color' : [],
                     'team_1_position' : [],
                     'team_2_color' : [],
                     'team_2_position' : [],
                     'misc_color' : [],
                     'misc_position' : [],
                     'misc_frequency' : [],
                     'misc_box_coord' : []}


frame_num = 0


for image_ad in image_batch:
    
    # Crop out the audiences via pitch segmentation
    rgb_image = cv2.imread(image_ad, cv2.COLOR_BGR2RGB)
    rgb_image = pitch_segmentation(rgb_image)

    # Run yolov8n on the current image and extract features from resulting boxes
    results = model.predict(rgb_image)
    boxes = results[0].boxes
    assignment, player_center_coord_list, box_coord_list = box_to_features(rgb_image, boxes)

    # Applies KNN with 3 clusters to find the most prominent colors as in rgb_color
    kmeans = KMeans(n_clusters=3)
    s=kmeans.fit(assignment)
    labels=kmeans.labels_

    # Extract the labels into 2 teams and misc color
    teams = Counter(labels).most_common(3)
    team_1_label = teams[0][0]
    team_2_label = teams[1][0]
    misc_label   = teams[2][0]

    #Return correct labels of (numbers) and the 2 team lab colors
    labels, lab_team_1_color, lab_team_2_color  = misc_in_teams(labels,assignment,teams)

    # Labeling via strings and consistency checks so that team X is always color Y:
    if frame_num == 0:
        team_1_color_global = lab_team_1_color 
        team_2_color_global = lab_team_2_color

        labels = list(map(lambda x: x if x != team_1_label else 'Team 1', labels))
        labels = list(map(lambda x: x if x != team_2_label else 'Team 2', labels))
        labels = list(map(lambda x: x if x != misc_label else 'Misc', labels))
        
    if frame_num > 0:
        global_1_vs_local_1 = delta_e_cie2000(team_1_color_global, lab_team_1_color)
        global_1_vs_local_2 = delta_e_cie2000(team_1_color_global, lab_team_2_color)
        if  global_1_vs_local_1 < global_1_vs_local_2:
            labels = list(map(lambda x: x if x != team_1_label else 'Team 1', labels))
            labels = list(map(lambda x: x if x != team_2_label else 'Team 2', labels))
        else: 
            labels = list(map(lambda x: x if x != team_2_label else 'Team 1', labels))
            labels = list(map(lambda x: x if x != team_1_label else 'Team 2', labels))
        labels = list(map(lambda x: x if x != misc_label else 'Misc', labels)) 

    # Updated teams indices
    team_1_idx = np.where(np.array(labels) == "Team 1")[0]
    team_2_idx = np.where(np.array(labels) == "Team 2")[0]
    misc_idx   = np.where(np.array(labels) == "Misc")[0]    
    
    # Update the misc color: 
    current_misc_box_coord = [box_coord_list[i] for i in misc_idx]
    current_misc_color = [assignment[i] for i in misc_idx]
    current_misc_lab_colors   = [rgb_to_lab_color(x) for x in current_misc_color]

    current_color_dict = {'player_center_coord_list': player_center_coord_list,
                          'current_misc_box_coord': current_misc_box_coord,
                          'current_misc_colors': current_misc_color,
                          'current_misc_lab_colors': current_misc_lab_colors,
                          'label': ["Misc"]*len(current_misc_color)}

    
    global_color_dict = update_misc_color(team_1_idx, team_2_idx, misc_idx, 
                                          global_color_dict, current_color_dict)
    
    
    frame_num +=1

    temp = rgb_image.copy()
    for i in range(len(box_coord_list)):
        box_coord = box_coord_list[i]
        





/home/huy-ng/Project/Football-analysis/.venv/lib/python3.8/site-packages/torch/cuda/__init__.py:88: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0

0: 384x640 21 persons, 1 ball, 232.9ms
Speed: 1.5ms preprocess, 232.9ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)
/home/huy-ng/Project/Football-analysis/.venv/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


NameError: name 'rgb_to_lab' is not defined